In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    ConfusionMatrixDisplay, roc_auc_score, RocCurveDisplay,
    f1_score, precision_score, recall_score,
)

In [2]:
X_train = pd.read_csv("telco-data/X_train.csv")
X_test = pd.read_csv("telco-data/X_test.csv")
y_train = pd.read_csv("telco-data/y_train.csv").squeeze("columns")
y_test = pd.read_csv("telco-data/y_test.csv").squeeze("columns")

print(f"X_train: {X_train.shape[0]:,} rows x {X_train.shape[1]} columns")
print(f"X_test:  {X_test.shape[0]:,} rows x {X_test.shape[1]} columns")
X_train.head()

X_train: 5,634 rows x 23 columns
X_test:  1,409 rows x 23 columns


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,...,PaperlessBilling,MonthlyCharges,InternetService_Fiber optic,InternetService_No,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,MultipleLines_No phone service,MultipleLines_Yes,tenure_group
0,0,0,1,0,3,1,0,0,0,0,...,1,74.10,1,0,1,0,0,0,1,0
1,1,0,1,0,6,1,0,0,0,0,...,0,25.10,0,1,0,0,1,0,1,0
2,1,0,0,0,31,1,0,1,1,0,...,1,103.45,1,0,0,1,0,0,1,2
3,1,0,1,1,47,1,1,1,1,0,...,1,96.10,1,0,0,1,0,0,0,2
4,0,1,0,0,11,1,0,0,0,0,...,0,89.70,1,0,0,0,0,0,0,0


In [3]:
# Sanity check, just checks that all outputs are 0 and 1, and makes sure that the y train and y test churn rates are similar
y_train = y_train.map({"Yes": 1, "No": 0})
y_test = y_test.map({"Yes": 1, "No": 0})

assert y_train.isna().sum() == 0 and y_test.isna().sum() == 0, "unexpected label found"

print("y_train churn rate:")
print(y_train.value_counts(normalize=True))
print("\ny_test churn rate:")
print(y_test.value_counts(normalize=True))

y_train churn rate:
Churn
0    0.734647
1    0.265353
Name: proportion, dtype: float64

y_test churn rate:
Churn
0    0.734564
1    0.265436
Name: proportion, dtype: float64


## Training model

In [4]:
rf = RandomForestClassifier(n_estimators=100, random_state=47, n_jobs=-1)
rf.fit(X_train, y_train)

RandomForestClassifier(n_jobs=-1, random_state=47)

## Making predictions

In [5]:
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

pd.DataFrame({"actual": y_test.values, "predicted": y_pred, "churn_probability": y_pred_proba.round(3)}).head(10)

,actual,predicted,churn_probability
0,0,0,0.16
1,1,1,0.64
2,0,1,0.67
3,0,0,0.03
4,0,1,0.64
5,1,0,0.28
6,0,0,0.00
7,1,0,0.41
8,1,0,0.28
9,0,0,0.00


## Evaluating model

### Accuracy (very baseline evaluation)

In [6]:
print(f"Accuracy: {accuracy_score(y_test, y_pred):.3f}")

Accuracy: 0.768


### w/o class_weight = balanced (for now)


##### Precision: measures our model's positive prediction, how correct our model overall. In our context, of everyone PREDICTED to churn, how many actually did? (and for the "No churn" row: of everyone predicted NOT to churn, how many actually didn't?) 
##### Recall: measures our model's ability to find all actual positives. In our context, of everyone who ACTUALLY churned, how many did we correctly predict? (and for the "No churn" row: of everyone actually did NOT churn, how many did we correctly predict as not churning?)
##### F1-score: measures the balance between precision and recall

In [7]:
print(classification_report(y_test, y_pred, target_names=["No churn", "Churn"]))

              precision    recall  f1-score   support

    No churn       0.81      0.89      0.85      1035
       Churn       0.58      0.44      0.50       374

    accuracy                           0.77      1409
   macro avg       0.70      0.66      0.68      1409
weighted avg       0.75      0.77      0.76      1409



### ROC-AUC and Confusion matrix 

In [8]:
print(f"ROC-AUC:  {roc_auc_score(y_test, y_pred_proba):.3f}\n")

ROC-AUC:  0.809



If we randomly picked one customer who actually churned and one who actually didn't, our model gives the customer who actually churned a higher predicted churn probability 81% of the time.

#### Plot

In [9]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(ax=axes[0], cmap="Blues", colorbar=False)
axes[0].set_title("Confusion Matrix")

RocCurveDisplay.from_predictions(y_test, y_pred_proba, ax=axes[1])
axes[1].set_title("ROC Curve")

plt.tight_layout()
plt.show()

/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/181737694.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


The model predicts no churn more accurately than churn. Since only 26.5% of training data are churners, our model would lean more to predicting no churn

### w/ class_weight = balanced

In [10]:
rf_balanced = RandomForestClassifier(
    n_estimators=100, random_state=47, n_jobs=-1, class_weight="balanced"
)
rf_balanced.fit(X_train, y_train)

y_pred_balanced = rf_balanced.predict(X_test)
y_pred_proba_balanced = rf_balanced.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_balanced):.3f}\n")
print(classification_report(y_test, y_pred_balanced, target_names=["No churn", "Churn"]))

ROC-AUC: 0.804

              precision    recall  f1-score   support

    No churn       0.81      0.88      0.85      1035
       Churn       0.57      0.43      0.49       374

    accuracy                           0.76      1409
   macro avg       0.69      0.65      0.67      1409
weighted avg       0.75      0.76      0.75      1409



In [11]:
cm_balanced = confusion_matrix(y_test, y_pred_balanced)
ConfusionMatrixDisplay(cm_balanced, display_labels=["No churn", "Churn"]).plot(cmap="Blues", colorbar=False)
plt.title("Confusion Matrix (Balanced)")
plt.show()

/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/3666488787.py:4: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Comparing `rf` vs `rf_balanced` (Churn class)

| Metric | `rf` (baseline) | `rf_balanced` | Change |
|---|---|---|---|
| Precision | 0.596 | 0.545 | ↓ 0.051 |
| Recall | 0.438 | 0.650 | ↑ 0.212 |
| F1-score | 0.505 | 0.592 | ↑ 0.087 |
| False Negatives | 210 | 131 | ↓ 79 |
| False Positives | 111 | 203 | ↑ 92 |

Balancing the classes catches a lot more real churners but there is more false alarms than before. So it is a tradeoff. However, missing a churner is worse than false alarm bc it is more costly. The company loses revenue if they lose customers with no chance to intervene if it doesn't know they are at risk of leaving.


### Overfitting check

In [12]:
train_acc = rf.score(X_train, y_train)
test_acc = rf.score(X_test, y_test)
print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")
print(f"Gap:            {train_acc - test_acc:.3f}")

Train accuracy: 0.997
Test accuracy:  0.768
Gap:            0.229


In [13]:
train_acc_b = rf_balanced.score(X_train, y_train)
test_acc_b = rf_balanced.score(X_test, y_test)
print(f"Train accuracy (balanced): {train_acc_b:.3f}")
print(f"Test accuracy (balanced):  {test_acc_b:.3f}")
print(f"Gap:                       {train_acc_b - test_acc_b:.3f}")


Train accuracy (balanced): 0.997
Test accuracy (balanced):  0.762
Gap:                       0.235


Both `rf` and `rf_balanced` **overfit** and show a large train/test accuracy gap (~0.22+), meaning trees are growing unconstrained and memorizing training data.

Next step tuning:
Possible fixes are constraining tree growth by `max_depth`, `min_samples_leaf`, `min_samples_split`, and/or tune `n_estimators` with cross-validation like `GridSearchCV` to close this gap.

## Hyperparameter tuning (GridSearchCV, maximize F1)

Search over tree-growth hyperparameters on **train only** with stratified 5-fold CV. Selection metric is **mean CV F1** only (`refit="f1"`). Precision, recall, accuracy, and ROC-AUC are tracked for tradeoff inspection, not for choosing the winner. Held-out `X_test` is reserved for a single final evaluation after params are frozen.

In [14]:
param_grid = {
    "max_depth": [5, 8, 10, 15, None],
    "min_samples_leaf": [1, 5, 10, 20],
    "min_samples_split": [2, 10, 20],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=47)

scoring = {
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
    "accuracy": "accuracy",
    "roc_auc": "roc_auc",
}

gs = GridSearchCV(
    estimator=RandomForestClassifier(
        n_estimators=100,
        class_weight="balanced",
        random_state=47,
        n_jobs=-1,
    ),
    param_grid=param_grid,
    scoring=scoring,
    refit="f1",  # ONLY this metric selects best_params_
    cv=cv,
    n_jobs=-1,
    return_train_score=True,
)

gs.fit(X_train, y_train)
rf_tuned = gs.best_estimator_

print("Best params:", gs.best_params_)
print(f"Best CV F1: {gs.best_score_:.3f}")


Best params: {'max_depth': 10, 'min_samples_leaf': 20, 'min_samples_split': 2}
Best CV F1: 0.639


### CV tradeoff table (top 10 by mean F1)

Sorted by `rank_test_f1`. Primary column is `mean_test_f1`; other metrics show what the F1 winner costs. Compare `mean_train_f1` vs `mean_test_f1` for a CV-level overfitting signal (especially unconstrained rows with `max_depth=None`, `min_samples_leaf=1`).

In [15]:
results = pd.DataFrame(gs.cv_results_)
cols = [
    "param_max_depth",
    "param_min_samples_leaf",
    "param_min_samples_split",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_accuracy",
    "mean_test_roc_auc",
    "mean_train_f1",
    "rank_test_f1",
]
top = results[cols].sort_values("rank_test_f1").head(10)
display(top)


,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_f1,std_test_f1,mean_test_precision,mean_test_recall,mean_test_accuracy,mean_test_roc_auc,mean_train_f1,rank_test_f1
33,10,20,2,0.638772,0.011909,0.541985,0.777926,0.766598,0.850620,0.667094,1
34,10,20,10,0.638772,0.011909,0.541985,0.777926,0.766598,0.850620,0.667094,1
35,10,20,20,0.638772,0.011909,0.541985,0.777926,0.766598,0.850620,0.667094,1
53,None,5,20,0.638670,0.018253,0.560837,0.741806,0.777426,0.848974,0.715981,4
14,8,1,20,0.638415,0.016874,0.546971,0.767224,0.769616,0.849595,0.692004,5
39,15,5,2,0.638183,0.015486,0.565221,0.733110,0.779555,0.847297,0.742878,6
40,15,5,10,0.638183,0.015486,0.565221,0.733110,0.779555,0.847297,0.742878,6
38,15,1,20,0.638018,0.018640,0.564639,0.733779,0.779201,0.847228,0.733464,8
50,None,1,20,0.637886,0.014913,0.565950,0.731104,0.779910,0.846970,0.734391,9
41,15,5,20,0.637624,0.018421,0.560408,0.739799,0.777071,0.848906,0.715545,10


### Evaluating `rf_tuned` (best CV F1 params, threshold 0.5)

Test metrics below are the honest estimate of the tuned forest at the **default 0.5** cutoff. CV F1 was only used to choose hyperparameters — not for this test look.

In [16]:
y_pred_tuned = rf_tuned.predict(X_test)
y_proba_tuned = rf_tuned.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_tuned):.3f}\n")
print(classification_report(y_test, y_pred_tuned, target_names=["No churn", "Churn"]))


Accuracy: 0.735
ROC-AUC:  0.831

              precision    recall  f1-score   support

    No churn       0.90      0.72      0.80      1035
       Churn       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.74      0.75      1409



In [17]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_tuned = confusion_matrix(y_test, y_pred_tuned)
ConfusionMatrixDisplay(cm_tuned, display_labels=["No churn", "Churn"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Confusion Matrix (Tuned)")

RocCurveDisplay.from_predictions(y_test, y_proba_tuned, ax=axes[1])
axes[1].set_title("ROC Curve (Tuned)")

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/4154604192.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Overfitting check (`rf_tuned`)

In [18]:
train_acc_t = rf_tuned.score(X_train, y_train)
test_acc_t = rf_tuned.score(X_test, y_test)
print(f"Train accuracy (tuned): {train_acc_t:.3f}")
print(f"Test accuracy  (tuned): {test_acc_t:.3f}")
print(f"Gap:                    {train_acc_t - test_acc_t:.3f}")

y_pred_train_t = rf_tuned.predict(X_train)
train_f1_t = f1_score(y_train, y_pred_train_t)
test_f1_t = f1_score(y_test, y_pred_tuned)
print(f"\nTrain F1 (tuned, 0.5): {train_f1_t:.3f}")
print(f"Test F1  (tuned, 0.5): {test_f1_t:.3f}")
print(f"F1 gap:                {train_f1_t - test_f1_t:.3f}")


Train accuracy (tuned): 0.785
Test accuracy  (tuned): 0.735
Gap:                    0.050

Train F1 (tuned, 0.5): 0.669
Test F1  (tuned, 0.5): 0.610
F1 gap:                0.059


### Comparing `rf` vs `rf_balanced` vs `rf_tuned` (Churn class)

All three models use threshold **0.5**. Selection for `rf_tuned` was by **CV F1 only**; columns below are for tradeoff reading.

In [19]:
def churn_metrics(y_true, y_pred, y_proba, model, X_tr, y_tr, X_te, y_te):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    return {
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1-score": f1_score(y_true, y_pred),
        "False Negatives": fn,
        "False Positives": fp,
        "Accuracy": accuracy_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
        "Train–test accuracy gap": model.score(X_tr, y_tr) - model.score(X_te, y_te),
    }

metrics_rf = churn_metrics(y_test, y_pred, y_pred_proba, rf, X_train, y_train, X_test, y_test)
metrics_bal = churn_metrics(
    y_test, y_pred_balanced, y_pred_proba_balanced, rf_balanced, X_train, y_train, X_test, y_test
)
metrics_tuned = churn_metrics(
    y_test, y_pred_tuned, y_proba_tuned, rf_tuned, X_train, y_train, X_test, y_test
)

comparison = pd.DataFrame(
    {
        "rf (0.5)": metrics_rf,
        "rf_balanced (0.5)": metrics_bal,
        "rf_tuned (0.5)": metrics_tuned,
    }
)
comparison["Change vs rf_balanced"] = comparison["rf_tuned (0.5)"] - comparison["rf_balanced (0.5)"]

display(comparison.round(3))
print("Best params (frozen from CV F1):", gs.best_params_)


,rf (0.5),rf_balanced (0.5),rf_tuned (0.5),Change vs rf_balanced
Precision,0.583,0.570,0.501,-0.069
Recall,0.441,0.425,0.781,0.356
F1-score,0.502,0.487,0.610,0.123
False Negatives,209.000,215.000,82.000,-133.000
False Positives,118.000,120.000,291.000,171.000
Accuracy,0.768,0.762,0.735,-0.027
ROC-AUC,0.809,0.804,0.831,0.027
Train–test accuracy gap,0.229,0.235,0.050,-0.185


Best params (frozen from CV F1): {'max_depth': 10, 'min_samples_leaf': 20, 'min_samples_split': 2}


**Interpretation notes**

- Hyperparameters for `rf_tuned` were chosen by maximizing **mean CV F1** only; precision, recall, accuracy, and ROC-AUC did not affect who won.
- Check whether regularization closed the train/test accuracy gap relative to `rf` / `rf_balanced` (~0.22+).
- Check whether churn recall / false negatives improved or worsened vs `rf_balanced`.
- Business preference still holds: missing a churner is costlier than a false alarm.
- If a nearby CV rank-2/3 config had much higher recall with only slightly lower F1, treat it as a business alternative — do **not** silently switch selection rules mid-notebook.

### Decision rule (documented)

- Hyperparameters were chosen by **5-fold stratified `GridSearchCV`** maximizing **mean CV F1**.
- Precision, recall, accuracy, and ROC-AUC were tracked during CV for tradeoff inspection only.
- `best_params_` were frozen before evaluating on `X_test`.
- Default threshold **0.5** remains for this section; threshold tuning (if done) is a separate follow-up on `rf_tuned` (see `rf-threshold-cv-plan.md`).
- `n_estimators` stayed fixed at 100 for this pass.

## Threshold tuning (cross-validation)

Choose decision thresholds with **5-fold stratified CV on train only**. The forest uses the frozen `rf_tuned` hyperparameters from GridSearchCV. Held-out `X_test` is used once after cutoffs are frozen.

Two operating points are selected from the same CV grid:

1. **`best_t`** — maximize **mean CV F1** (balanced precision/recall).
2. **`best_t_prec`** — maximize **mean CV precision** subject to **mean CV recall ≥ `RECALL_FLOOR`** (default `0.65`). This is the FP-reduction lever: higher precision usually means fewer false alarms while keeping churn catch-rate above the floor.

Raising the cutoff above 0.5 is the main way to cut false positives from `rf_tuned` @ 0.5.


In [20]:
# CV threshold search on train only, using frozen rf_tuned growth params
thresholds = np.round(np.arange(0.1, 0.9, 0.01), 2)
cv_thresh = StratifiedKFold(n_splits=5, shuffle=True, random_state=47)

# Change this to tighten/loosen the precision-constrained pick
RECALL_FLOOR = 0.65

fold_f1 = {t: [] for t in thresholds}
fold_precision = {t: [] for t in thresholds}
fold_recall = {t: [] for t in thresholds}

# Same growth params as rf_tuned; refit each fold (no leakage from full-train rf_tuned)
tuned_kwargs = {
    "n_estimators": 100,
    "class_weight": "balanced",
    "random_state": 47,
    "n_jobs": -1,
    **gs.best_params_,
}

for train_idx, val_idx in cv_thresh.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    fold_model = RandomForestClassifier(**tuned_kwargs)
    fold_model.fit(X_tr, y_tr)
    val_proba = fold_model.predict_proba(X_val)[:, 1]

    for t in thresholds:
        y_hat = (val_proba >= t).astype(int)
        fold_f1[t].append(f1_score(y_val, y_hat))
        fold_precision[t].append(precision_score(y_val, y_hat, zero_division=0))
        fold_recall[t].append(recall_score(y_val, y_hat, zero_division=0))

cv_threshold_results = pd.DataFrame(
    {
        "threshold": thresholds,
        "mean_cv_f1": [np.mean(fold_f1[t]) for t in thresholds],
        "mean_cv_precision": [np.mean(fold_precision[t]) for t in thresholds],
        "mean_cv_recall": [np.mean(fold_recall[t]) for t in thresholds],
    }
)

# --- Pick 1: max mean CV F1 ---
best_row = cv_threshold_results.loc[cv_threshold_results["mean_cv_f1"].idxmax()]
best_t = float(best_row["threshold"])

print("=== F1-maximizing threshold ===")
print(f"Best threshold (max mean CV F1): {best_t:.2f}")
print(f"Mean CV F1:        {best_row['mean_cv_f1']:.3f}")
print(f"Mean CV precision: {best_row['mean_cv_precision']:.3f}")
print(f"Mean CV recall:    {best_row['mean_cv_recall']:.3f}")

# --- Pick 2: max mean CV precision s.t. mean CV recall >= RECALL_FLOOR ---
feasible = cv_threshold_results[
    cv_threshold_results["mean_cv_recall"] >= RECALL_FLOOR
].copy()

if feasible.empty:
    raise ValueError(
        f"No threshold met RECALL_FLOOR={RECALL_FLOOR}. "
        "Lower RECALL_FLOOR or inspect cv_threshold_results."
    )

# Highest precision; tie-break toward higher threshold (fewer predicted positives / fewer FPs)
feasible = feasible.sort_values(
    ["mean_cv_precision", "threshold"], ascending=[False, False]
)
best_row_prec = feasible.iloc[0]
best_t_prec = float(best_row_prec["threshold"])

print(f"\n=== Precision-constrained threshold (recall >= {RECALL_FLOOR:.2f}) ===")
print(f"Best threshold (max mean CV precision): {best_t_prec:.2f}")
print(f"Mean CV F1:        {best_row_prec['mean_cv_f1']:.3f}")
print(f"Mean CV precision: {best_row_prec['mean_cv_precision']:.3f}")
print(f"Mean CV recall:    {best_row_prec['mean_cv_recall']:.3f}")
print(f"Feasible thresholds: {len(feasible)} / {len(cv_threshold_results)}")


=== F1-maximizing threshold ===
Best threshold (max mean CV F1): 0.50
Mean CV F1:        0.639
Mean CV precision: 0.542
Mean CV recall:    0.778

=== Precision-constrained threshold (recall >= 0.65) ===
Best threshold (max mean CV precision): 0.61
Mean CV F1:        0.634
Mean CV precision: 0.618
Mean CV recall:    0.651
Feasible thresholds: 52 / 80


### CV threshold tradeoff table / plot

Full grid of mean CV precision, recall, and F1. Vertical lines mark **`best_t`** (F1) and **`best_t_prec`** (max precision with recall ≥ floor).


In [21]:
display(cv_threshold_results.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cv_threshold_results["threshold"], cv_threshold_results["mean_cv_f1"], label="Mean CV F1")
ax.plot(
    cv_threshold_results["threshold"],
    cv_threshold_results["mean_cv_precision"],
    label="Mean CV Precision",
    alpha=0.8,
)
ax.plot(
    cv_threshold_results["threshold"],
    cv_threshold_results["mean_cv_recall"],
    label="Mean CV Recall",
    alpha=0.8,
)
ax.axhline(RECALL_FLOOR, color="gray", linestyle=":", alpha=0.8, label=f"recall floor = {RECALL_FLOOR:.2f}")
ax.axvline(best_t, color="black", linestyle="--", label=f"best_t (F1) = {best_t:.2f}")
ax.axvline(best_t_prec, color="tab:red", linestyle="--", label=f"best_t_prec = {best_t_prec:.2f}")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Threshold CV curves (train folds only)")
ax.legend()
plt.tight_layout()
plt.show()


,threshold,mean_cv_f1,mean_cv_precision,mean_cv_recall
0,0.10,0.486,0.322,0.992
1,0.11,0.493,0.328,0.992
2,0.12,0.498,0.333,0.990
3,0.13,0.503,0.337,0.988
4,0.14,0.508,0.342,0.985
...,...,...,...,...
75,0.85,0.307,0.821,0.189
76,0.86,0.280,0.849,0.168
77,0.87,0.246,0.857,0.144
78,0.88,0.195,0.877,0.110


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/1402664869.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Freeze thresholds and apply on test

`rf_thresholded` reuses the frozen `rf_tuned` forest (already fit on full `X_train`). Only the decision cutoffs change. Test labels are not used to choose `best_t` or `best_t_prec`.


In [22]:
rf_thresholded = rf_tuned  # frozen growth params; already fit on full train
test_proba_thresh = rf_thresholded.predict_proba(X_test)[:, 1]

y_test_hat = (test_proba_thresh >= best_t).astype(int)
y_test_hat_prec = (test_proba_thresh >= best_t_prec).astype(int)

print(f"Frozen best_t (F1)              = {best_t:.2f}")
print(f"Frozen best_t_prec (prec@recall)= {best_t_prec:.2f}  [floor={RECALL_FLOOR:.2f}]")
print(f"Predicted churn rate @ 0.50:         {(test_proba_thresh >= 0.5).mean():.3f}")
print(f"Predicted churn rate @ best_t:       {y_test_hat.mean():.3f}")
print(f"Predicted churn rate @ best_t_prec:  {y_test_hat_prec.mean():.3f}")


Frozen best_t (F1)              = 0.50
Frozen best_t_prec (prec@recall)= 0.61  [floor=0.65]
Predicted churn rate @ 0.50:         0.414
Predicted churn rate @ best_t:       0.414
Predicted churn rate @ best_t_prec:  0.303


### Evaluating `rf_thresholded` @ best_t (F1)

Test metrics for the **F1-maximizing** cutoff. CV F1 chose `best_t`; this is the honest test look for that operating point. ROC-AUC is threshold-independent and should match `rf_tuned`.


In [23]:
print(f"Accuracy: {accuracy_score(y_test, y_test_hat):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, test_proba_thresh):.3f}\n")
print(classification_report(y_test, y_test_hat, target_names=["No churn", "Churn"]))


Accuracy: 0.735
ROC-AUC:  0.831

              precision    recall  f1-score   support

    No churn       0.90      0.72      0.80      1035
       Churn       0.50      0.78      0.61       374

    accuracy                           0.74      1409
   macro avg       0.70      0.75      0.70      1409
weighted avg       0.79      0.74      0.75      1409



In [24]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_thresh = confusion_matrix(y_test, y_test_hat)
ConfusionMatrixDisplay(cm_thresh, display_labels=["No churn", "Churn"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title(f"Confusion Matrix (threshold={best_t:.2f})")

RocCurveDisplay.from_predictions(y_test, test_proba_thresh, ax=axes[1])
axes[1].set_title(f"ROC Curve (thresholded @ {best_t:.2f})")

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/2432927940.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Evaluating `rf_thresholded` @ best_t_prec (precision-constrained)

Test metrics for the **max precision subject to recall ≥ floor** cutoff. This is the FP-focused operating point. Same forest / same probabilities as above — only the threshold differs.


In [25]:
print(f"Accuracy: {accuracy_score(y_test, y_test_hat_prec):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, test_proba_thresh):.3f}\n")
print(classification_report(y_test, y_test_hat_prec, target_names=["No churn", "Churn"]))


Accuracy: 0.779
ROC-AUC:  0.831

              precision    recall  f1-score   support

    No churn       0.87      0.82      0.85      1035
       Churn       0.57      0.66      0.61       374

    accuracy                           0.78      1409
   macro avg       0.72      0.74      0.73      1409
weighted avg       0.79      0.78      0.78      1409



In [26]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_thresh_prec = confusion_matrix(y_test, y_test_hat_prec)
ConfusionMatrixDisplay(cm_thresh_prec, display_labels=["No churn", "Churn"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title(f"Confusion Matrix (prec-constrained t={best_t_prec:.2f})")

RocCurveDisplay.from_predictions(y_test, test_proba_thresh, ax=axes[1])
axes[1].set_title(f"ROC Curve (same ranking; t={best_t_prec:.2f})")

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/2754825289.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Comparing operating points (Churn class)

Same test set and forest ranking. Columns:

- `rf` / `rf_balanced` / `rf_tuned` @ **0.5**
- `rf_thresholded` @ **`best_t`** (max CV F1)
- `rf_thresholded` @ **`best_t_prec`** (max CV precision with recall ≥ floor)

**Change vs rf_tuned** uses the precision-constrained cutoff — the FP-reduction pick.


In [27]:
def churn_metrics_at_threshold(y_true, y_hat, y_proba):
    tn, fp, fn, tp = confusion_matrix(y_true, y_hat).ravel()
    return {
        "Precision": precision_score(y_true, y_hat),
        "Recall": recall_score(y_true, y_hat),
        "F1-score": f1_score(y_true, y_hat),
        "False Negatives": fn,
        "False Positives": fp,
        "Accuracy": accuracy_score(y_true, y_hat),
        "ROC-AUC": roc_auc_score(y_true, y_proba),
    }

metrics_rf_thr = churn_metrics_at_threshold(y_test, y_pred, y_pred_proba)
metrics_bal_thr = churn_metrics_at_threshold(y_test, y_pred_balanced, y_pred_proba_balanced)
metrics_tuned_thr = churn_metrics_at_threshold(y_test, y_pred_tuned, y_proba_tuned)
metrics_best_t = churn_metrics_at_threshold(y_test, y_test_hat, test_proba_thresh)
metrics_best_t_prec = churn_metrics_at_threshold(y_test, y_test_hat_prec, test_proba_thresh)

col_f1 = f"thresh F1 ({best_t:.2f})"
col_prec = f"thresh prec ({best_t_prec:.2f})"
comparison_thresh = pd.DataFrame(
    {
        "rf (0.5)": metrics_rf_thr,
        "rf_balanced (0.5)": metrics_bal_thr,
        "rf_tuned (0.5)": metrics_tuned_thr,
        col_f1: metrics_best_t,
        col_prec: metrics_best_t_prec,
    }
)
comparison_thresh["Change vs rf_tuned (prec)"] = (
    comparison_thresh[col_prec] - comparison_thresh["rf_tuned (0.5)"]
)

display(comparison_thresh.round(3))
print(f"best_t (F1)               = {best_t:.2f}")
print(f"best_t_prec (prec@recall) = {best_t_prec:.2f}  [RECALL_FLOOR={RECALL_FLOOR:.2f}]")
print("Best params (unchanged):", gs.best_params_)


,rf (0.5),rf_balanced (0.5),rf_tuned (0.5),thresh F1 (0.50),thresh prec (0.61),Change vs rf_tuned (prec)
Precision,0.583,0.570,0.501,0.501,0.574,0.073
Recall,0.441,0.425,0.781,0.781,0.655,-0.126
F1-score,0.502,0.487,0.610,0.610,0.612,0.001
False Negatives,209.000,215.000,82.000,82.000,129.000,47.000
False Positives,118.000,120.000,291.000,291.000,182.000,-109.000
Accuracy,0.768,0.762,0.735,0.735,0.779,0.044
ROC-AUC,0.809,0.804,0.831,0.831,0.831,0.000


best_t (F1)               = 0.50
best_t_prec (prec@recall) = 0.61  [RECALL_FLOOR=0.65]
Best params (unchanged): {'max_depth': 10, 'min_samples_leaf': 20, 'min_samples_split': 2}


In [28]:
# Side-by-side confusion matrices for FN / FP comparison
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

cms = [
    (confusion_matrix(y_test, y_pred), "rf (0.5)"),
    (confusion_matrix(y_test, y_pred_balanced), "rf_balanced (0.5)"),
    (confusion_matrix(y_test, y_pred_tuned), "rf_tuned (0.5)"),
    (cm_thresh, f"F1 t={best_t:.2f}"),
    (cm_thresh_prec, f"prec t={best_t_prec:.2f}"),
]

for ax, (cm, title) in zip(axes, cms):
    ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/700162923.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation notes**

- **`best_t`** maximizes mean CV F1; **`best_t_prec`** maximizes mean CV precision among thresholds with mean CV recall ≥ `RECALL_FLOOR`.
- Compare FPs for `rf_tuned` @ 0.5 vs the precision-constrained column — that is the intended FP reduction.
- If `best_t` and `best_t_prec` coincide, the F1 winner already sat on the recall-floor frontier; try a lower `RECALL_FLOOR` (e.g. `0.60`) for a more aggressive FP cut.
- Missing a churner is still costlier than a false alarm; the recall floor encodes how much catch-rate you refuse to give up.
- ROC-AUC should match `rf_tuned` (same ranking). Accuracy may move opposite to recall; expected tradeoff.


### Overfitting check (both frozen thresholds)

Train/test metrics use the **same frozen cutoffs** (not `model.score()`, which always uses 0.5). Threshold tuning does not by itself fix unconstrained-tree overfitting; that was addressed by `rf_tuned` growth constraints.


In [29]:
train_proba_thresh = rf_thresholded.predict_proba(X_train)[:, 1]
train_hat_f1 = (train_proba_thresh >= best_t).astype(int)
train_hat_prec = (train_proba_thresh >= best_t_prec).astype(int)

print(f"--- F1 threshold t={best_t:.2f} ---")
print(f"Train accuracy: {accuracy_score(y_train, train_hat_f1):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_test_hat):.3f}")
print(f"Gap:            {accuracy_score(y_train, train_hat_f1) - accuracy_score(y_test, y_test_hat):.3f}")
print(f"Train F1: {f1_score(y_train, train_hat_f1):.3f} | Test F1: {f1_score(y_test, y_test_hat):.3f}")

print(f"\n--- Precision-constrained t={best_t_prec:.2f} (floor={RECALL_FLOOR:.2f}) ---")
print(f"Train accuracy: {accuracy_score(y_train, train_hat_prec):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_test_hat_prec):.3f}")
print(f"Gap:            {accuracy_score(y_train, train_hat_prec) - accuracy_score(y_test, y_test_hat_prec):.3f}")
print(f"Train F1: {f1_score(y_train, train_hat_prec):.3f} | Test F1: {f1_score(y_test, y_test_hat_prec):.3f}")
print(f"Train precision: {precision_score(y_train, train_hat_prec):.3f} | Test precision: {precision_score(y_test, y_test_hat_prec):.3f}")
print(f"Train recall:    {recall_score(y_train, train_hat_prec):.3f} | Test recall:    {recall_score(y_test, y_test_hat_prec):.3f}")


--- F1 threshold t=0.50 ---
Train accuracy: 0.785
Test accuracy:  0.735
Gap:            0.050
Train F1: 0.669 | Test F1: 0.610

--- Precision-constrained t=0.61 (floor=0.65) ---
Train accuracy: 0.814
Test accuracy:  0.779
Gap:            0.035
Train F1: 0.662 | Test F1: 0.612
Train precision: 0.640 | Test precision: 0.574
Train recall:    0.686 | Test recall:    0.655


### Decision rule (documented) — threshold

- Two cutoffs were chosen by **5-fold stratified CV** on `X_train` / `y_train` over `np.arange(0.1, 0.9, 0.01)`:
  - **`best_t`**: maximize **mean CV F1**.
  - **`best_t_prec`**: maximize **mean CV precision** among thresholds with **mean CV recall ≥ `RECALL_FLOOR`** (default `0.65`); ties → higher threshold.
- Search used the frozen `rf_tuned` hyperparameters (`gs.best_params_`); fold models were refit each fold to avoid leakage.
- Both cutoffs were frozen before evaluating on `X_test`.
- Default **0.5** remains the reference for earlier cells (`rf`, `rf_balanced`, `rf_tuned`).
- Comparison tables use the same test set and churn-class metric definitions as earlier sections.


## Alternate path: unweighted baseline — tune then threshold

Explore the same two levers already used on the class-weighted forest — **growth hyperparameter CV** then **threshold CV** — but start from the **baseline** `RandomForestClassifier` with **no** `class_weight="balanced"`.

- Same `param_grid`, stratified 5-fold CV (`random_state=47`), multi-metric scoring, and `refit="f1"` as the weighted path.
- Same threshold grid and the same `RECALL_FLOOR` already defined above.
- Only intentional training change: **omit** `class_weight="balanced"` (default / `None`).
- Existing weighted cells stay intact (`rf_balanced` → `rf_tuned` → thresholds). This section is an ablation / alternate system.

| Object | Name |
|---|---|
| Grid search | `gs_base` |
| Tuned forest | `rf_tuned_base` |
| F1 threshold | `best_t_base` |
| Precision@floor threshold | `best_t_prec_base` |
| Test preds @ F1 t | `y_test_hat_base` |
| Test preds @ prec t | `y_test_hat_prec_base` |


### Hyperparameter search (unweighted GridSearchCV)

Reuse the same growth grid and CV settings as `gs`. Selection metric remains **mean CV F1** only.


In [30]:
gs_base = GridSearchCV(
    estimator=RandomForestClassifier(
        n_estimators=100,
        # class_weight intentionally omitted (None)
        random_state=47,
        n_jobs=-1,
    ),
    param_grid=param_grid,  # same dict as weighted search
    scoring=scoring,        # f1, precision, recall, accuracy, roc_auc
    refit="f1",
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=47),
    n_jobs=-1,
    return_train_score=True,
)

gs_base.fit(X_train, y_train)
rf_tuned_base = gs_base.best_estimator_

print("Best params (unweighted):", gs_base.best_params_)
print(f"Best CV F1 (unweighted): {gs_base.best_score_:.3f}")


Best params (unweighted): {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best CV F1 (unweighted): 0.576


### CV tradeoff table (top 10 by mean F1) — unweighted

Sorted by `rank_test_f1`. Primary column is `mean_test_f1`; other metrics are for tradeoff reading only.


In [31]:
results_base = pd.DataFrame(gs_base.cv_results_)
cols_base = [
    "param_max_depth",
    "param_min_samples_leaf",
    "param_min_samples_split",
    "mean_test_f1",
    "std_test_f1",
    "mean_test_precision",
    "mean_test_recall",
    "mean_test_accuracy",
    "mean_test_roc_auc",
    "mean_train_f1",
    "rank_test_f1",
]
top_base = results_base[cols_base].sort_values("rank_test_f1").head(10)
display(top_base)


,param_max_depth,param_min_samples_leaf,param_min_samples_split,mean_test_f1,std_test_f1,mean_test_precision,mean_test_recall,mean_test_accuracy,mean_test_roc_auc,mean_train_f1,rank_test_f1
28,10,5,10,0.576178,0.030235,0.677285,0.502341,0.804229,0.848143,0.670439,1
27,10,5,2,0.576178,0.030235,0.677285,0.502341,0.804229,0.848143,0.670439,1
40,15,5,10,0.575917,0.028732,0.673145,0.504348,0.803341,0.847185,0.693582,3
39,15,5,2,0.575917,0.028732,0.673145,0.504348,0.803341,0.847185,0.693582,3
26,10,1,20,0.575257,0.030131,0.675397,0.501672,0.803696,0.847842,0.663491,5
52,None,5,10,0.574886,0.027920,0.669237,0.505017,0.802275,0.847163,0.695746,6
51,None,5,2,0.574886,0.027920,0.669237,0.505017,0.802275,0.847163,0.695746,6
41,15,5,20,0.573817,0.028829,0.675539,0.499666,0.803341,0.848133,0.662320,8
56,None,10,20,0.573558,0.030673,0.684714,0.494314,0.805293,0.848528,0.636007,9
55,None,10,10,0.573558,0.030673,0.684714,0.494314,0.805293,0.848528,0.636007,9


### Evaluating `rf_tuned_base` (best CV F1 params, threshold 0.5)

Test metrics for the unweighted tuned forest at the **default 0.5** cutoff. Expect relatively lower recall / fewer FPs than `rf_tuned` @ 0.5 — imbalance is not handled in training on this path.


In [32]:
y_pred_tuned_base = rf_tuned_base.predict(X_test)
y_proba_tuned_base = rf_tuned_base.predict_proba(X_test)[:, 1]

print(f"Accuracy: {accuracy_score(y_test, y_pred_tuned_base):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, y_proba_tuned_base):.3f}\n")
print(classification_report(y_test, y_pred_tuned_base, target_names=["No churn", "Churn"]))


Accuracy: 0.789
ROC-AUC:  0.830

              precision    recall  f1-score   support

    No churn       0.83      0.90      0.86      1035
       Churn       0.63      0.49      0.55       374

    accuracy                           0.79      1409
   macro avg       0.73      0.69      0.71      1409
weighted avg       0.78      0.79      0.78      1409



In [33]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_tuned_base = confusion_matrix(y_test, y_pred_tuned_base)
ConfusionMatrixDisplay(cm_tuned_base, display_labels=["No churn", "Churn"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title("Confusion Matrix (Unweighted Tuned)")

RocCurveDisplay.from_predictions(y_test, y_proba_tuned_base, ax=axes[1])
axes[1].set_title("ROC Curve (Unweighted Tuned)")

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/363030903.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Overfitting check (`rf_tuned_base`)

Compare the train/test gap to raw `rf` (~0.23) and weighted `rf_tuned` (~0.07). Regularization alone should shrink the gap even without class weights.


In [34]:
train_acc_tb = rf_tuned_base.score(X_train, y_train)
test_acc_tb = rf_tuned_base.score(X_test, y_test)
print(f"Train accuracy (unweighted tuned): {train_acc_tb:.3f}")
print(f"Test accuracy  (unweighted tuned): {test_acc_tb:.3f}")
print(f"Gap:                               {train_acc_tb - test_acc_tb:.3f}")

y_pred_train_tb = rf_tuned_base.predict(X_train)
train_f1_tb = f1_score(y_train, y_pred_train_tb)
test_f1_tb = f1_score(y_test, y_pred_tuned_base)
print(f"\nTrain F1 (unweighted tuned, 0.5): {train_f1_tb:.3f}")
print(f"Test F1  (unweighted tuned, 0.5): {test_f1_tb:.3f}")
print(f"F1 gap:                           {train_f1_tb - test_f1_tb:.3f}")


Train accuracy (unweighted tuned): 0.842
Test accuracy  (unweighted tuned): 0.789
Gap:                               0.053

Train F1 (unweighted tuned, 0.5): 0.663
Test F1  (unweighted tuned, 0.5): 0.553
F1 gap:                           0.110


### Threshold CV on frozen `rf_tuned_base` params

Same procedure as the weighted threshold section: 5-fold stratified CV on train only, threshold grid `np.arange(0.1, 0.9, 0.01)`, and the **same** `RECALL_FLOOR`.

Two operating points:

1. **`best_t_base`** — maximize mean CV F1.
2. **`best_t_prec_base`** — maximize mean CV precision subject to mean CV recall ≥ `RECALL_FLOOR`.

**Hypothesis:** both cutoffs land **below 0.5** (opposite direction from weighted `best_t` / `best_t_prec`).


In [35]:
# CV threshold search on train only, using frozen rf_tuned_base growth params
thresholds_base = np.round(np.arange(0.1, 0.9, 0.01), 2)
cv_thresh_base = StratifiedKFold(n_splits=5, shuffle=True, random_state=47)

fold_f1_base = {t: [] for t in thresholds_base}
fold_precision_base = {t: [] for t in thresholds_base}
fold_recall_base = {t: [] for t in thresholds_base}

# Same growth params as rf_tuned_base; no class_weight; refit each fold
tuned_kwargs_base = {
    "n_estimators": 100,
    "random_state": 47,
    "n_jobs": -1,
    **gs_base.best_params_,
}

for train_idx, val_idx in cv_thresh_base.split(X_train, y_train):
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    fold_model = RandomForestClassifier(**tuned_kwargs_base)
    fold_model.fit(X_tr, y_tr)
    val_proba = fold_model.predict_proba(X_val)[:, 1]

    for t in thresholds_base:
        y_hat = (val_proba >= t).astype(int)
        fold_f1_base[t].append(f1_score(y_val, y_hat))
        fold_precision_base[t].append(precision_score(y_val, y_hat, zero_division=0))
        fold_recall_base[t].append(recall_score(y_val, y_hat, zero_division=0))

cv_threshold_results_base = pd.DataFrame(
    {
        "threshold": thresholds_base,
        "mean_cv_f1": [np.mean(fold_f1_base[t]) for t in thresholds_base],
        "mean_cv_precision": [np.mean(fold_precision_base[t]) for t in thresholds_base],
        "mean_cv_recall": [np.mean(fold_recall_base[t]) for t in thresholds_base],
    }
)

# --- Pick 1: max mean CV F1 ---
best_row_base = cv_threshold_results_base.loc[cv_threshold_results_base["mean_cv_f1"].idxmax()]
best_t_base = float(best_row_base["threshold"])

print("=== F1-maximizing threshold (unweighted) ===")
print(f"Best threshold (max mean CV F1): {best_t_base:.2f}")
print(f"Mean CV F1:        {best_row_base['mean_cv_f1']:.3f}")
print(f"Mean CV precision: {best_row_base['mean_cv_precision']:.3f}")
print(f"Mean CV recall:    {best_row_base['mean_cv_recall']:.3f}")

# --- Pick 2: max mean CV precision s.t. mean CV recall >= RECALL_FLOOR ---
feasible_base = cv_threshold_results_base[
    cv_threshold_results_base["mean_cv_recall"] >= RECALL_FLOOR
].copy()

if feasible_base.empty:
    raise ValueError(
        f"No unweighted threshold met RECALL_FLOOR={RECALL_FLOOR}. "
        "Lower RECALL_FLOOR or inspect cv_threshold_results_base."
    )

# Highest precision; tie-break toward higher threshold (fewer predicted positives / fewer FPs)
best_row_prec_base = feasible_base.sort_values(
    ["mean_cv_precision", "threshold"], ascending=[False, False]
).iloc[0]
best_t_prec_base = float(best_row_prec_base["threshold"])

print(f"\n=== Precision-constrained threshold (unweighted, recall >= {RECALL_FLOOR:.2f}) ===")
print(f"Best threshold (max mean CV precision): {best_t_prec_base:.2f}")
print(f"Mean CV F1:        {best_row_prec_base['mean_cv_f1']:.3f}")
print(f"Mean CV precision: {best_row_prec_base['mean_cv_precision']:.3f}")
print(f"Mean CV recall:    {best_row_prec_base['mean_cv_recall']:.3f}")
print(f"Feasible thresholds: {len(feasible_base)} / {len(cv_threshold_results_base)}")


=== F1-maximizing threshold (unweighted) ===
Best threshold (max mean CV F1): 0.32
Mean CV F1:        0.640
Mean CV precision: 0.561
Mean CV recall:    0.744

=== Precision-constrained threshold (unweighted, recall >= 0.65) ===
Best threshold (max mean CV precision): 0.38
Mean CV F1:        0.630
Mean CV precision: 0.601
Mean CV recall:    0.664
Feasible thresholds: 29 / 80


### CV threshold tradeoff table / plot — unweighted

Full grid of mean CV precision, recall, and F1. Vertical lines mark **`best_t_base`** (F1) and **`best_t_prec_base`** (max precision with recall ≥ floor).


In [36]:
display(cv_threshold_results_base.round(3))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(cv_threshold_results_base["threshold"], cv_threshold_results_base["mean_cv_f1"], label="Mean CV F1")
ax.plot(
    cv_threshold_results_base["threshold"],
    cv_threshold_results_base["mean_cv_precision"],
    label="Mean CV precision",
)
ax.plot(
    cv_threshold_results_base["threshold"],
    cv_threshold_results_base["mean_cv_recall"],
    label="Mean CV recall",
)
ax.axhline(RECALL_FLOOR, color="gray", linestyle=":", alpha=0.8, label=f"recall floor = {RECALL_FLOOR:.2f}")
ax.axvline(best_t_base, color="black", linestyle="--", label=f"best_t_base (F1) = {best_t_base:.2f}")
ax.axvline(best_t_prec_base, color="tab:red", linestyle="--", label=f"best_t_prec_base = {best_t_prec_base:.2f}")
ax.axvline(0.5, color="tab:green", linestyle=":", alpha=0.7, label="0.50 reference")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Unweighted threshold CV curves (train folds only)")
ax.legend()
plt.tight_layout()
plt.show()


,threshold,mean_cv_f1,mean_cv_precision,mean_cv_recall
0,0.10,0.548,0.384,0.957
1,0.11,0.555,0.392,0.950
2,0.12,0.560,0.399,0.940
3,0.13,0.566,0.406,0.933
4,0.14,0.573,0.416,0.924
...,...,...,...,...
75,0.85,0.047,0.969,0.024
76,0.86,0.031,0.982,0.016
77,0.87,0.020,0.975,0.010
78,0.88,0.015,0.800,0.007


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/4091281692.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Freeze unweighted thresholds and apply on test

Reuse `rf_tuned_base` (already fit on full `X_train`). Only the decision cutoffs change. Test labels are not used to choose `best_t_base` or `best_t_prec_base`.


In [37]:
test_proba_base = rf_tuned_base.predict_proba(X_test)[:, 1]

y_test_hat_base = (test_proba_base >= best_t_base).astype(int)
y_test_hat_prec_base = (test_proba_base >= best_t_prec_base).astype(int)

print(f"Frozen best_t_base (F1)              = {best_t_base:.2f}")
print(f"Frozen best_t_prec_base (prec@recall)= {best_t_prec_base:.2f}  [floor={RECALL_FLOOR:.2f}]")
print(f"Predicted churn rate @ 0.50:              {(test_proba_base >= 0.5).mean():.3f}")
print(f"Predicted churn rate @ best_t_base:       {y_test_hat_base.mean():.3f}")
print(f"Predicted churn rate @ best_t_prec_base:  {y_test_hat_prec_base.mean():.3f}")


Frozen best_t_base (F1)              = 0.32
Frozen best_t_prec_base (prec@recall)= 0.38  [floor=0.65]
Predicted churn rate @ 0.50:              0.207
Predicted churn rate @ best_t_base:       0.375
Predicted churn rate @ best_t_prec_base:  0.315


### Evaluating unweighted path @ `best_t_base` (F1)

Test metrics for the **F1-maximizing** unweighted cutoff. ROC-AUC is threshold-independent and should match `rf_tuned_base`.


In [38]:
print(f"Accuracy: {accuracy_score(y_test, y_test_hat_base):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, test_proba_base):.3f}\n")
print(classification_report(y_test, y_test_hat_base, target_names=["No churn", "Churn"]))


Accuracy: 0.752
ROC-AUC:  0.830

              precision    recall  f1-score   support

    No churn       0.89      0.76      0.82      1035
       Churn       0.52      0.74      0.61       374

    accuracy                           0.75      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.79      0.75      0.76      1409



In [39]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_base_f1 = confusion_matrix(y_test, y_test_hat_base)
ConfusionMatrixDisplay(cm_base_f1, display_labels=["No churn", "Churn"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title(f"Confusion Matrix (unweighted t={best_t_base:.2f})")

RocCurveDisplay.from_predictions(y_test, test_proba_base, ax=axes[1])
axes[1].set_title(f"ROC Curve (unweighted @ {best_t_base:.2f})")

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/2973033996.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Evaluating unweighted path @ `best_t_prec_base` (precision-constrained)

Test metrics for the **max precision subject to recall ≥ floor** unweighted cutoff. Same forest / same probabilities — only the threshold differs.


In [40]:
print(f"Accuracy: {accuracy_score(y_test, y_test_hat_prec_base):.3f}")
print(f"ROC-AUC:  {roc_auc_score(y_test, test_proba_base):.3f}\n")
print(classification_report(y_test, y_test_hat_prec_base, target_names=["No churn", "Churn"]))


Accuracy: 0.771
ROC-AUC:  0.830

              precision    recall  f1-score   support

    No churn       0.87      0.81      0.84      1035
       Churn       0.56      0.66      0.61       374

    accuracy                           0.77      1409
   macro avg       0.71      0.74      0.72      1409
weighted avg       0.79      0.77      0.78      1409



In [41]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

cm_base_prec = confusion_matrix(y_test, y_test_hat_prec_base)
ConfusionMatrixDisplay(cm_base_prec, display_labels=["No churn", "Churn"]).plot(
    ax=axes[0], cmap="Blues", colorbar=False
)
axes[0].set_title(f"Confusion Matrix (unweighted prec t={best_t_prec_base:.2f})")

RocCurveDisplay.from_predictions(y_test, test_proba_base, ax=axes[1])
axes[1].set_title(f"ROC Curve (unweighted ranking; t={best_t_prec_base:.2f})")

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/1862259389.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Overfitting check (unweighted frozen thresholds)

Train/test metrics use the **same frozen cutoffs** (not `model.score()`, which always uses 0.5).


In [42]:
train_proba_base = rf_tuned_base.predict_proba(X_train)[:, 1]
train_hat_f1_base = (train_proba_base >= best_t_base).astype(int)
train_hat_prec_base = (train_proba_base >= best_t_prec_base).astype(int)

print(f"--- Unweighted F1 threshold t={best_t_base:.2f} ---")
print(f"Train accuracy: {accuracy_score(y_train, train_hat_f1_base):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_test_hat_base):.3f}")
print(f"Gap:            {accuracy_score(y_train, train_hat_f1_base) - accuracy_score(y_test, y_test_hat_base):.3f}")
print(f"Train F1: {f1_score(y_train, train_hat_f1_base):.3f} | Test F1: {f1_score(y_test, y_test_hat_base):.3f}")

print(f"\n--- Unweighted precision-constrained t={best_t_prec_base:.2f} (floor={RECALL_FLOOR:.2f}) ---")
print(f"Train accuracy: {accuracy_score(y_train, train_hat_prec_base):.3f}")
print(f"Test accuracy:  {accuracy_score(y_test, y_test_hat_prec_base):.3f}")
print(f"Gap:            {accuracy_score(y_train, train_hat_prec_base) - accuracy_score(y_test, y_test_hat_prec_base):.3f}")
print(f"Train F1: {f1_score(y_train, train_hat_prec_base):.3f} | Test F1: {f1_score(y_test, y_test_hat_prec_base):.3f}")
print(f"Train precision: {precision_score(y_train, train_hat_prec_base):.3f} | Test precision: {precision_score(y_test, y_test_hat_prec_base):.3f}")
print(f"Train recall:    {recall_score(y_train, train_hat_prec_base):.3f} | Test recall:    {recall_score(y_test, y_test_hat_prec_base):.3f}")


--- Unweighted F1 threshold t=0.32 ---
Train accuracy: 0.826
Test accuracy:  0.752
Gap:            0.074
Train F1: 0.716 | Test F1: 0.614

--- Unweighted precision-constrained t=0.38 (floor=0.65) ---
Train accuracy: 0.840
Test accuracy:  0.771
Gap:            0.069
Train F1: 0.715 | Test F1: 0.606
Train precision: 0.679 | Test precision: 0.559
Train recall:    0.756 | Test recall:    0.663


### Head-to-head: weighted path vs unweighted baseline path

Same `X_test` / `y_test`. Columns cover the original baseline, both tuned @ 0.5 forests, and both threshold policies on each path. Prefer comparing the two precision@floor operating points when asking which system better controls FPs while holding catch-rate.


In [43]:
# Reuse churn_metrics_at_threshold from the weighted threshold section
metrics_h2h = {
    "rf (0.5)": churn_metrics_at_threshold(y_test, y_pred, y_pred_proba),
    "rf_tuned_base (0.5)": churn_metrics_at_threshold(
        y_test, y_pred_tuned_base, y_proba_tuned_base
    ),
    f"base F1 ({best_t_base:.2f})": churn_metrics_at_threshold(
        y_test, y_test_hat_base, test_proba_base
    ),
    f"base prec ({best_t_prec_base:.2f})": churn_metrics_at_threshold(
        y_test, y_test_hat_prec_base, test_proba_base
    ),
    "rf_tuned (0.5)": churn_metrics_at_threshold(y_test, y_pred_tuned, y_proba_tuned),
    f"weighted F1 ({best_t:.2f})": churn_metrics_at_threshold(
        y_test, y_test_hat, test_proba_thresh
    ),
    f"weighted prec ({best_t_prec:.2f})": churn_metrics_at_threshold(
        y_test, y_test_hat_prec, test_proba_thresh
    ),
}

comparison_h2h = pd.DataFrame(metrics_h2h)
col_base_prec = f"base prec ({best_t_prec_base:.2f})"
col_wt_prec = f"weighted prec ({best_t_prec:.2f})"
comparison_h2h["Δ base_prec − wt_prec"] = (
    comparison_h2h[col_base_prec] - comparison_h2h[col_wt_prec]
)

display(comparison_h2h.round(3))
print("Unweighted best params:", gs_base.best_params_)
print("Weighted best params:  ", gs.best_params_)
print(f"Thresholds — base F1/prec: {best_t_base:.2f} / {best_t_prec_base:.2f}")
print(f"Thresholds — wt   F1/prec: {best_t:.2f} / {best_t_prec:.2f}")
print(f"RECALL_FLOOR (shared): {RECALL_FLOOR:.2f}")


,rf (0.5),rf_tuned_base (0.5),base F1 (0.32),base prec (0.38),rf_tuned (0.5),weighted F1 (0.50),weighted prec (0.61),Δ base_prec − wt_prec
Precision,0.583,0.630,0.524,0.559,0.501,0.501,0.574,-0.015
Recall,0.441,0.492,0.741,0.663,0.781,0.781,0.655,0.008
F1-score,0.502,0.553,0.614,0.606,0.610,0.610,0.612,-0.005
False Negatives,209.000,190.000,97.000,126.000,82.000,82.000,129.000,-3.000
False Positives,118.000,108.000,252.000,196.000,291.000,291.000,182.000,14.000
Accuracy,0.768,0.789,0.752,0.771,0.735,0.735,0.779,-0.008
ROC-AUC,0.809,0.830,0.830,0.830,0.831,0.831,0.831,-0.001


Unweighted best params: {'max_depth': 10, 'min_samples_leaf': 5, 'min_samples_split': 2}
Weighted best params:   {'max_depth': 10, 'min_samples_leaf': 20, 'min_samples_split': 2}
Thresholds — base F1/prec: 0.32 / 0.38
Thresholds — wt   F1/prec: 0.50 / 0.61
RECALL_FLOOR (shared): 0.65


In [44]:
# Side-by-side CMs for the two precision@floor production candidates
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

cms_h2h = [
    (cm_thresh_prec, f"weighted prec t={best_t_prec:.2f}"),
    (cm_base_prec, f"base prec t={best_t_prec_base:.2f}"),
]

for ax, (cm, title) in zip(axes, cms_h2h):
    ConfusionMatrixDisplay(cm, display_labels=["No churn", "Churn"]).plot(
        ax=ax, cmap="Blues", colorbar=False
    )
    ax.set_title(title)

plt.tight_layout()
plt.show()


/var/folders/m4/3v615nl12x73_b9cv3jw2pyr0000gn/T/ipykernel_29547/589343202.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Interpretation notes (baseline path vs weighted path)**

Results from the head-to-head table (same `X_test` / `y_test`; selection was train-CV only):

- **Overfitting gap closed.** Unweighted tuning cut the train/test accuracy gap from raw `rf` (~0.23) to **`rf_tuned_base` ~0.05**, essentially matching weighted `rf_tuned` (~0.05). Regularization works without class weights.
- **Thresholds moved below 0.5 as hypothesized.** `best_t_base = 0.32`, `best_t_prec_base = 0.38`. Weighted cutoffs were `best_t = 0.50`, `best_t_prec = 0.61`. Same levers, opposite side of 0.5 (weighted F1 pick landed on the default cutoff this run).
- **@ 0.5, unweighted stays FP-conservative / recall-weak.** `rf_tuned_base` recall ~0.49 / FP 108 vs `rf_tuned` recall ~0.78 / FP 291. Hyperparams alone do not fix imbalance on the baseline path.
- **At matched `RECALL_FLOOR` (precision-constrained picks):**
  - `base prec (0.38)`: precision 0.559, recall 0.663, F1 0.606, FN 126, FP 196, ROC-AUC 0.830
  - `weighted prec (0.61)`: precision 0.574, recall 0.655, F1 0.612, FN 129, FP 182, ROC-AUC 0.831
  - ROC-AUC is essentially tied. Weighted has **fewer FPs** and slightly better precision/F1 at nearly the same recall; baseline has 3 fewer FNs.
- **Preferred production system: keep the weighted path** (`rf_tuned` + `best_t_prec`), with the unweighted path retained as the ablation. Ranking quality is comparable, but at the shared recall floor the weighted operating point controls false alarms better. Baseline+lower-threshold is a viable alternative if you prefer not to use `class_weight` and are willing to accept ~14 more FPs for a tiny FN gain.
- Missing churners is still costlier than false alarms; `RECALL_FLOOR = 0.65` encodes the catch-rate we will not give up when comparing paths.


### Decision rule (documented) — unweighted baseline path

- Unweighted growth params (`gs_base.best_params_`) were chosen by **5-fold stratified `GridSearchCV`** maximizing **mean CV F1** (`refit="f1"`).
- Thresholds were chosen by **5-fold stratified CV** on train only over `np.arange(0.1, 0.9, 0.01)`:
  - **`best_t_base`**: maximize **mean CV F1**.
  - **`best_t_prec_base`**: maximize **mean CV precision** among thresholds with **mean CV recall ≥ `RECALL_FLOOR`** (same floor as weighted path); ties → higher threshold.
- **`class_weight` was not used** on this path (intentionally omitted / `None`).
- All selection on train only; test evaluated once per frozen system (`rf_tuned_base` @ 0.5, @ `best_t_base`, @ `best_t_prec_base`).
- Weighted path (`rf_tuned` + thresholds) remains available for comparison; production choice is a documented decision from the head-to-head table, not an implicit overwrite of earlier cells.
- `n_estimators` stayed fixed at 100; same `param_grid` / CV folds / scoring as the weighted GridSearch.


## Model summary — which system for which situation

Project preference: **missing a churner (FN) is costlier than a false alarm (FP)**, but FPs still matter because each FP triggers unnecessary **outreach** (contacting a customer flagged as at-risk — email, call, discount, etc.). Metrics below are from the same held-out test set after train-only CV selection.

### Quick chooser (with justification)

| Situation | Best fit | Why |
|---|---|---|
| **Best fit now** (default production) | **Weighted prec @ 0.61** (`rf_tuned` + `best_t_prec`) | Hits the recall floor (~0.66) while cutting FPs vs `rf_tuned` @ 0.5 (182 vs 291). Best precision among high-recall options (~0.57). Same strong ROC-AUC (~0.83) as the weighted forest. Matches “catch most churners, don’t flood outreach.” |
| **Outreach is cheap / max save-rate** | **`rf_tuned` @ 0.5** | Highest recall (~0.78), fewest FNs (~82). Accept ~291 FPs when contacting extras is cheap and lost customers are expensive. |
| **Ban `class_weight`** | **Baseline prec @ 0.38** (`rf_tuned_base` + `best_t_prec_base`) | Same recall-floor policy without training weights. Competitive with weighted prec (recall ~0.66, F1 ~0.61); slightly more FPs (196 vs 182). Threshold carries the imbalance correction. |
| **Outreach is expensive / high-confidence only** | **`rf` or `rf_tuned_base` @ 0.5** | FP-conservative (≈108–118 FPs) and higher precision @ 0.5. Prefer `rf_tuned_base` if you also want the overfitting gap closed (~0.05 vs ~0.23 for raw `rf`). Expect low recall (~0.44–0.49). |
| **Keep for experiments / explainability** | **`rf_balanced`**; **both F1 thresholds** | `rf_balanced` isolates class weights before regularization (ablation). F1 thresholds (`best_t` / `best_t_base`) are the balanced-metric defaults when you don’t want a recall-floor policy. |

### Each system at a glance

#### 1. `rf` — unconstrained baseline @ 0.5
- **Best at:** Clean “before tuning” reference; ultra-conservative alerts.
- **Strengths:** Few FPs; simple to explain.
- **Weaknesses:** Misses most churners (recall ~0.44); large train/test gap (~0.23).
- **Future use:** Benchmark, or high-confidence-only lists when outreach is very expensive.

#### 2. `rf_balanced` — class weights, unconstrained @ 0.5
- **Best at:** Showing what imbalance handling alone does before growth tuning / thresholding.
- **Strengths:** Isolates the `class_weight` effect; useful ablation.
- **Weaknesses:** Still overfits; not a production candidate once `rf_tuned` exists.
- **Future use:** Teaching / explainability path in the notebook.

#### 3. `rf_tuned` — weighted + growth-tuned @ 0.5
- **Best at:** Maximum catch-rate retention screen.
- **Strengths:** Highest recall (~0.78); gap closed (~0.05); strong ROC-AUC (~0.83).
- **Weaknesses:** Many FPs (~291); precision ~0.50 → alert fatigue if used as-is.
- **Future use:** Wide-net campaigns when outreach is cheap.

#### 4. Weighted F1 threshold (`best_t` ≈ 0.50)
- **Best at:** Single balanced P/R operating point under max CV F1 (same as `rf_tuned` @ 0.5 on this run).
- **Strengths:** Documented CV F1 selection rule.
- **Weaknesses:** No extra FP control vs default 0.5 here.
- **Future use:** Default when you want F1, not a recall-floor policy.

#### 5. Weighted prec@floor (`best_t_prec` ≈ 0.61) — **current production pick**
- **Best at:** Project default — catch most churners while limiting false outreach.
- **Strengths:** Recall above floor (~0.66); fewer FPs than @ 0.5; best precision among high-recall picks.
- **Weaknesses:** Gives up some catch-rate vs @ 0.5 (FN 129 vs 82); needs a chosen `RECALL_FLOOR`.
- **Future use:** Primary prioritized outreach list with an explicit catch-rate guarantee.

#### 6. `rf_tuned_base` — unweighted + growth-tuned @ 0.5
- **Best at:** FP-conservative tuned forest; “fit then set policy later.”
- **Strengths:** Gap closed; solid ROC-AUC; highest precision @ 0.5 (~0.63).
- **Weaknesses:** Recall still weak @ 0.5 (~0.49) until the threshold moves.
- **Future use:** Ranking/scoring service, or when training must stay unweighted.

#### 7. Baseline F1 threshold (`best_t_base` ≈ 0.32)
- **Best at:** Buying recall without class weights (aggressive unweighted cutoff).
- **Strengths:** Strong recall (~0.74) with no `class_weight`.
- **Weaknesses:** Many FPs (~252); low cutoff can be harder to interpret.
- **Future use:** Policy bans class weights but still needs high catch-rate.

#### 8. Baseline prec@floor (`best_t_prec_base` ≈ 0.38)
- **Best at:** Unweighted alternative to weighted prec at the same recall floor.
- **Strengths:** Hits ~0.66 recall without weights; competitive F1.
- **Weaknesses:** Slightly more FPs / slightly worse precision than weighted prec.
- **Future use:** Fallback production system if `class_weight` is dropped.

### How to pick for this project

1. **Default now:** weighted prec @ 0.61.  
2. Shift to `rf_tuned` @ 0.5 only if outreach cost is low and max save-rate is the goal.  
3. Shift to baseline prec @ 0.38 only if class weights are disallowed.  
4. Shift to `rf` / `rf_tuned_base` @ 0.5 only if outreach must stay high-confidence and low volume.
